# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/hands-on-multimodal-AI/blob/main/hands-on/HANDS_ON_session_03_Extract_Audio_from_Video_QA.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://www.oreilly.com/library/view/transformers-the-definitive/9781098167004/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="Transformers: The Definitive Guide"/>
</a>




<font color="red" size="10">
<b>HANDS-ON TIME: 15 mins</b>
</font>

# About this Notebook

When working with video files, **you don't always need the visual stream**. If your task is purely audio-based, transcription, speaker analysis, sound-event detection, or audio Q&A, loading full video frames wastes GPU memory and slows inference.

This notebook demonstrates a practical **video → audio → Q&A** pipeline:

| Step | What you do |
|---|---|
| **1. Extract audio with FFmpeg** | Strip the audio track from an MP4 video — no video decoding required |
| **2. Load Qwen2-Audio-7B-Instruct** | A powerful audio-language model that can answer natural-language questions about audio |
| **3. Single-file Audio Q&A** | Ask questions about the extracted audio clip |
| **4. Batch Audio Q&A** | Process multiple audio files in a single forward pass for throughput |
| **5. Production tips** | Format choices, chunking long audio, memory management, and error handling |

### Why extract audio first?
- **Memory**: A 1-minute 1080p video ≈ 1.5 GB decoded frames vs. ~10 MB of 16 kHz mono WAV audio.
- **Speed**: Skipping video decode can cut pre-processing time by 10–50×.
- **Simplicity**: Audio-only models are smaller and faster than full video-language models.

## Install Dependencies


In [ ]:
!pip install --upgrade transformers==5.2.0 accelerate==1.12.0 librosa==0.11.0 gdown==5.2.1 -q

## Download the Video from Google Drive

We'll grab an MP4 video file hosted on Google Drive using `gdown`. The file ID is extracted from the sharing URL. After download we verify the file exists and print its size — this is the video we'll extract audio from.

In [ ]:
import os

# Google Drive file ID from the sharing URL
file_id = "17ZeTnfVoCC-uEioa5uQgTwPR_DFeD5Cd"
video_path = "input_video.mp4"

!gdown --id {file_id} -O {video_path}

if os.path.exists(video_path):
    size_mb = os.path.getsize(video_path) / (1024 * 1024)
    print(f"✅ Video downloaded: {video_path}  ({size_mb:.1f} MB)")
else:
    raise FileNotFoundError("Download failed — check the Google Drive link and sharing permissions.")

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=17ZeTnfVoCC-uEioa5uQgTwPR_DFeD5Cd
To: /content/input_video.mp4
100% 285k/285k [00:00<00:00, 128MB/s]
✅ Video downloaded: input_video.mp4  (0.3 MB)


## Extract Audio with FFmpeg

FFmpeg is the industry-standard tool for media manipulation. Below we use it to **strip the audio track out of the video** without re-encoding the video stream — this is fast and lossless.

### Production-ready FFmpeg flags explained

| Flag | What it does | Why it matters |
|---|---|---|
| `-vn` | Drop all video streams | Saves memory & disk — we only need audio |
| `-acodec pcm_s16le` | Encode audio as 16-bit PCM (WAV) | Uncompressed format preserves quality for ML models |
| `-ar 16000` | Resample to 16 kHz | Matches the sample rate expected by most speech models (Whisper, Qwen2-Audio, etc.) |
| `-ac 1` | Mix down to mono | Most speech models expect single-channel audio; stereo doubles memory for no benefit |
| `-y` | Overwrite output without asking | Essential for non-interactive / CI pipelines |

> **Tip**: In production, you could swap `pcm_s16le` for `flac` to get **~50% smaller files** with zero quality loss (FLAC is lossless). Use `-acodec flac` instead.

In [ ]:
import subprocess, os

audio_path = "extracted_audio.wav"

cmd = [
    "ffmpeg",
    "-i", video_path,       # input video
    "-vn",                   # drop video
    "-acodec", "pcm_s16le",  # 16-bit PCM WAV
    "-ar", "16000",          # 16 kHz sample rate
    "-ac", "1",              # mono
    "-y",                    # overwrite without prompt
    audio_path,
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print("❌ FFmpeg error:\n", result.stderr)
else:
    audio_size_mb = os.path.getsize(audio_path) / (1024 * 1024)
    video_size_mb = os.path.getsize(video_path) / (1024 * 1024)
    print(f"✅ Audio extracted: {audio_path}")
    print(f"   Video size:  {video_size_mb:.1f} MB")
    print(f"   Audio size:  {audio_size_mb:.1f} MB")
    print(f"   Reduction:   {(1 - audio_size_mb / video_size_mb) * 100:.0f}%")

✅ Audio extracted: extracted_audio.wav
   Video size:  0.3 MB
   Audio size:  0.2 MB
   Reduction:   40%


## Preview the Extracted Audio

Quick sanity check — play the extracted WAV file inline and display its duration and sample rate to make sure FFmpeg did its job correctly.

In [ ]:
import librosa
from IPython.display import Audio, display

# Load with librosa to get duration info
y, sr = librosa.load(audio_path, sr=None)
duration_sec = len(y) / sr

print(f"Sample rate : {sr} Hz")
print(f"Duration    : {duration_sec:.1f} seconds ({duration_sec/60:.1f} min)")
print(f"Samples     : {len(y):,}")

display(Audio(audio_path, autoplay=False))

Sample rate : 16000 Hz
Duration    : 5.4 seconds (0.1 min)
Samples     : 86,016


## Load Qwen2-Audio-7B-Instruct

**Qwen2-Audio-7B-Instruct** is a 7-billion-parameter audio-language model from Alibaba. It accepts audio + text prompts and generates natural-language answers — think of it as "ChatGPT that can listen."

- **`AutoProcessor`** converts raw audio waveforms into the mel-spectrogram features the model expects, and tokenizes the text prompt.
- **`Qwen2AudioForConditionalGeneration`** is the model itself. We use `device_map="auto"` to distribute layers across available GPUs automatically.

> **Production note**: For serving, consider quantising with `load_in_4bit=True` (via `bitsandbytes`) to cut VRAM usage roughly in half with minimal quality loss.

In [ ]:
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration

model_id = "Qwen/Qwen2-Audio-7B-Instruct"

processor = AutoProcessor.from_pretrained(model_id)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    model_id, device_map="auto"
)

print(f"Model loaded on: {model.device}")
print(f"Model dtype   : {model.dtype}")

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/853 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/876 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

Model loaded on: cuda:0
Model dtype   : torch.bfloat16


## Audio Q&A on the Extracted Audio

Now let's ask the model questions about the audio we extracted from the video. We build a chat-style conversation with:
- An **audio** element pointing to our local WAV file.
- A **text** element containing the question.

The processor's `apply_chat_template` formats this into the token sequence the model expects, and `librosa` loads the audio at the processor's required sample rate. We then generate and decode the answer.

In [ ]:
conversation1 = [
    {"role": "user", "content": [
        {"type": "audio", "audio_url": "/content/extracted_audio.wav"},
        {"type": "text", "text": "Describe what is happening in this audio. What are the main topics discussed?"},
    ]},
]

conversation2 = [
    {"role": "user", "content": [
        {"type": "audio", "audio_url": "/content/extracted_audio.wav"},
        {"type": "text", "text": "Can you guess the speaker's sentiment?"},
    ]},
]

conversations = [conversation1, conversation2]


In [ ]:
from io import BytesIO
from urllib.request import urlopen
# Prepare inputs
text = [processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False) for conv in conversations]

sr = processor.feature_extractor.sampling_rate
audios = []

for conversation in conversations:
    for message in conversation:
        if isinstance(message["content"], list):
            for ele in message["content"]:
                if ele["type"] == "audio":
                    audio_path = ele['audio_url']
                    if audio_path.startswith("http"):
                        # Remote URL
                        audio, _ = librosa.load(BytesIO(urlopen(audio_path).read()), sr=sr)
                    else:
                        # Load local file
                        if not os.path.exists(audio_path):
                            raise FileNotFoundError(f"Local audio file not found: {audio_path}")
                        audio, _ = librosa.load(audio_path, sr=sr)
                    audios.append(audio)

# Batch processing
inputs = processor(text=text, audio=audios, return_tensors="pt", padding=True)

# Move all tensor inputs to the correct device
device = model.device
inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

# Generate response
generate_ids = model.generate(**inputs, max_new_tokens=512)
generate_ids = generate_ids[:, inputs["input_ids"].size(1):]

# Decode
responses = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

# Print outputs
for i, r in enumerate(responses):
    print(f"Response {i+1}: {r}")


It is strongly recommended to pass the `sampling_rate` argument to `WhisperFeatureExtractor()`. Failing to do so can result in silent errors that might be hard to debug.


Response 1: The audio describes a hands-on, multi-model AI live course.
Response 2: The speaker is excited.


# Production Tips & Tricks

Below are practical patterns you'd use when moving an audio-Q&A pipeline from a notebook into a **production service**.

### Chunk long audio files
Most speech models have a context-length limit (e.g. ~30 s for Whisper, ~30 min for Qwen2-Audio). For very long recordings, split the audio into overlapping chunks, process each one, then merge the answers.

In [ ]:
import numpy as np, librosa

def chunk_audio(path, chunk_sec=30, overlap_sec=2, target_sr=16000):
    """
    Split an audio file into overlapping chunks.

    Parameters
    ----------
    path        : str   – path to audio file
    chunk_sec   : int   – length of each chunk in seconds
    overlap_sec : int   – overlap between consecutive chunks (helps avoid cutting mid-word)
    target_sr   : int   – resample to this rate

    Returns
    -------
    list[np.ndarray] – list of audio arrays (float32, mono)
    """
    y, sr = librosa.load(path, sr=target_sr)
    chunk_samples = chunk_sec * sr
    overlap_samples = overlap_sec * sr
    step = chunk_samples - overlap_samples

    chunks = []
    start = 0
    while start < len(y):
        end = min(start + chunk_samples, len(y))
        chunks.append(y[start:end])
        start += step

    print(f"Split {len(y)/sr:.1f}s audio into {len(chunks)} chunks "
          f"({chunk_sec}s each, {overlap_sec}s overlap)")
    return chunks

# Demo: chunk the extracted audio
chunks = chunk_audio(audio_path, chunk_sec=30, overlap_sec=2)
for i, c in enumerate(chunks):
    print(f"  Chunk {i}: {len(c)/16000:.1f}s")

Split 5.4s audio into 1 chunks (30s each, 2s overlap)
  Chunk 0: 5.4s


### FFmpeg format comparison for storage/quality trade-offs

In production you'll want to choose the right output format based on your constraints. Here's a quick comparison you can run to see the file-size differences between WAV, FLAC, and MP3.

In [ ]:
import subprocess, os

formats = {
    "WAV (PCM 16-bit)":  {"ext": "wav",  "codec": "pcm_s16le"},
    "FLAC (lossless)":   {"ext": "flac", "codec": "flac"},
    "MP3 (128 kbps)":    {"ext": "mp3",  "codec": "libmp3lame",  "extra": ["-b:a", "128k"]},
}

print(f"{'Format':<22} {'Size':>10}  Notes")
print("-" * 55)

for name, cfg in formats.items():
    out = f"_compare.{cfg['ext']}"
    cmd = [
        "ffmpeg", "-i", video_path, "-vn",
        "-acodec", cfg["codec"], "-ar", "16000", "-ac", "1",
    ] + cfg.get("extra", []) + ["-y", out]

    subprocess.run(cmd, capture_output=True, text=True)
    size_kb = os.path.getsize(out) / 1024
    print(f"{name:<22} {size_kb:>8.0f} KB")
    os.remove(out)   # clean up

print("\n💡 FLAC gives ~50% compression with zero quality loss — best for ML pipelines.")
print("   MP3 is smallest but lossy — fine for playback, may hurt model accuracy slightly.")

Format                       Size  Notes
-------------------------------------------------------
WAV (PCM 16-bit)            168 KB
FLAC (lossless)             175 KB
MP3 (128 kbps)               86 KB

💡 FLAC gives ~50% compression with zero quality loss — best for ML pipelines.
   MP3 is smallest but lossy — fine for playback, may hurt model accuracy slightly.


### GPU Memory Management

Large audio-language models consume significant VRAM. In a production service you'll want to:
- **Delete the model** when you're done with a batch to free memory for the next task.
- **Clear the CUDA cache** so PyTorch releases the GPU memory back to the driver.
- **Use `torch.inference_mode()`** (or `torch.no_grad()`) to avoid storing computation graphs during inference.

In [ ]:
import torch, gc

def print_gpu_memory(tag=""):
    """Print current GPU memory usage (works on CUDA-enabled machines)."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved  = torch.cuda.memory_reserved()  / 1024**3
        print(f"[{tag}] GPU memory — allocated: {allocated:.2f} GB, reserved: {reserved:.2f} GB")
    else:
        print(f"[{tag}] No CUDA device available — running on CPU")

print_gpu_memory("Before cleanup")

# ── Clean up after inference ──
# In a real service you'd do this between requests or when switching models:
del model
del processor
gc.collect()
torch.cuda.empty_cache()


# For this notebook we keep the model loaded, but here's the pattern:
print("\n📌 Production pattern:")
print("   del model; del processor")
print("   gc.collect()")
print("   torch.cuda.empty_cache()")

print_gpu_memory("After cleanup")

[Before cleanup] GPU memory — allocated: 15.65 GB, reserved: 31.36 GB

📌 Production pattern:
   del model; del processor
   gc.collect()
   torch.cuda.empty_cache()
[After cleanup] GPU memory — allocated: 0.01 GB, reserved: 15.66 GB


### Robust audio extraction helper for production

Below is a production-ready Python function that wraps the FFmpeg call with:
- **Input validation** — checks the video file exists before running.
- **Configurable parameters** — sample rate, channels, codec, and output format are all arguments.
- **Error handling** — raises clear exceptions on failure with the FFmpeg stderr included.
- **Probe metadata** — optionally returns duration and codec info from `ffprobe`.

In [ ]:
import subprocess, os, json
from pathlib import Path

def extract_audio(
    video_path: str,
    output_path: str = None,
    sample_rate: int = 16000,
    channels: int = 1,
    codec: str = "pcm_s16le",
    output_format: str = "wav",
) -> str:
    """
    Extract audio from a video file using FFmpeg.

    Parameters
    ----------
    video_path    : Path to input video.
    output_path   : Path for output audio (auto-generated if None).
    sample_rate   : Target sample rate in Hz.
    channels      : Number of audio channels (1 = mono).
    codec         : Audio codec (pcm_s16le, flac, libmp3lame, etc.).
    output_format : File extension / container format.

    Returns
    -------
    str : Path to the extracted audio file.
    """
    if not os.path.isfile(video_path):
        raise FileNotFoundError(f"Video not found: {video_path}")

    if output_path is None:
        stem = Path(video_path).stem
        output_path = f"{stem}_audio.{output_format}"

    cmd = [
        "ffmpeg",
        "-i", video_path,
        "-vn",
        "-acodec", codec,
        "-ar", str(sample_rate),
        "-ac", str(channels),
        "-y",
        output_path,
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        raise RuntimeError(
            f"FFmpeg failed (exit code {result.returncode}):\n{result.stderr}"
        )

    return output_path


def probe_audio(file_path: str) -> dict:
    """Return metadata (duration, codec, sample rate, etc.) via ffprobe."""
    cmd = [
        "ffprobe", "-v", "quiet",
        "-print_format", "json",
        "-show_streams", "-show_format",
        file_path,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffprobe failed:\n{result.stderr}")
    return json.loads(result.stdout)


# ── Demo: extract + probe ──
out = extract_audio(video_path, output_path="demo_extracted.wav")
meta = probe_audio(out)

audio_stream = next(s for s in meta["streams"] if s["codec_type"] == "audio")
duration = float(meta["format"]["duration"])

print(f" Extracted: {out}")
print(f"   Codec      : {audio_stream['codec_name']}")
print(f"   Sample rate : {audio_stream['sample_rate']} Hz")
print(f"   Channels    : {audio_stream['channels']}")
print(f"   Duration    : {duration:.1f}s ({duration/60:.1f} min)")
print(f"   File size   : {os.path.getsize(out)/1024:.0f} KB")

# Clean up the demo file
os.remove(out)

 Extracted: demo_extracted.wav
   Codec      : pcm_s16le
   Sample rate : 16000 Hz
   Channels    : 1
   Duration    : 5.4s (0.1 min)
   File size   : 168 KB
